In [ ]:
from __future__ import annotations

import math
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
torch.set_grad_enabled(False)

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.data_sources import load_array_from_path
from src.models_macro import Parellel_Renorm_Dynamic as MacroParellelRenormDynamic
from src.models_micro import Parellel_Renorm_Dynamic as MicroParellelRenormDynamic

DEFAULT_RUN_NAME = "stage2_macro"
DEFAULT_DATA_PATH = Path("loc_data_kuramoto") / "generated_data.npz"


class CompatibleMacroParellelRenormDynamic(MacroParellelRenormDynamic):
    """Allow loading checkpoints whose first retained macro scale keeps the micro dimension."""

    def _build_scale_dims(self, reduce_dims):
        reduce_dim_schedule = self._resolve_reduce_dims(reduce_dims)
        if len(reduce_dim_schedule) == 1 and int(reduce_dim_schedule[0]) == self.sym_size:
            return (
                [self.sym_size],
                [{"type": "dense", "input_dim": self.sym_size, "output_dim": self.sym_size}],
                reduce_dim_schedule,
            )
        return super()._build_scale_dims(reduce_dims)


In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the causal_network_mix_2_0.2_syn2 project root.")


def parse_reduce_dims(value) -> List[int]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def extract_index(path: Path, pattern: str) -> Optional[int]:
    match = re.search(pattern, path.name)
    return None if match is None else int(match.group(1))


def list_stage2_scale_artifacts(project_root: Path, run_name: str = DEFAULT_RUN_NAME) -> List[Dict]:
    model_dir = project_root / "loc_model_stage2" / run_name
    result_dir = project_root / "loc_result_stage2" / run_name
    if not model_dir.exists():
        raise FileNotFoundError(f"Missing model directory: {model_dir}")
    if not result_dir.exists():
        raise FileNotFoundError(f"Missing result directory: {result_dir}")

    model_map = {
        idx: path
        for path in sorted(model_dir.glob("model_scale*.pkl"))
        for idx in [extract_index(path, r"model_scale(\d+)\.pkl")]
        if idx is not None
    }
    summary_map = {
        idx: path
        for path in sorted(result_dir.glob("summary_scale*.csv"))
        for idx in [extract_index(path, r"summary_scale(\d+)\.csv")]
        if idx is not None
    }

    artifacts = []
    for file_scale in sorted(set(model_map) | set(summary_map)):
        model_path = model_map.get(file_scale)
        summary_path = summary_map.get(file_scale)
        if model_path is None or summary_path is None:
            continue

        summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
        scale_dims = parse_reduce_dims(summary_row.get("scale_dims", ""))
        logical_scale_id = int(summary_row.get("scale_id", file_scale))
        macro_dim = None
        if scale_dims and 0 <= logical_scale_id < len(scale_dims):
            macro_dim = int(scale_dims[logical_scale_id])

        artifacts.append(
            {
                "file_scale": int(file_scale),
                "model_family": "micro" if int(file_scale) == 0 else "macro",
                "logical_scale_id": logical_scale_id,
                "macro_dim": macro_dim,
                "hidden_units1": int(summary_row.get("hidden_units1", 100)),
                "hidden_units2": int(summary_row.get("hidden_units2", 100)),
                "flow_num_layers": int(summary_row.get("flow_num_layers", 3)),
                "dynamics_num_layers": int(summary_row.get("dynamics_num_layers", 4)),
                "latent_size": int(summary_row.get("latent_size", 1)),
                "time_delay": int(summary_row.get("time_delay", 1)),
                "reduce_dims": parse_reduce_dims(summary_row.get("reduce_dims", "")),
                "model_path": model_path,
                "summary_path": summary_path,
            }
        )

    if not artifacts:
        raise FileNotFoundError("No paired stage2 macro artifacts were found.")
    return artifacts


def load_stage2_origin_data(project_root: Path, data_path: Optional[Path] = None) -> np.ndarray:
    resolved_path = project_root / (data_path or DEFAULT_DATA_PATH)
    origin_data = load_array_from_path(str(resolved_path))
    if origin_data is None:
        raise FileNotFoundError(f"Could not load origin data from: {resolved_path}")
    return np.asarray(origin_data, dtype=np.float32)


def build_prediction_model(num_nodes: int, artifact: Dict, device: Union[str, torch.device] = "cpu"):
    device = torch.device(device)
    model_cls = MicroParellelRenormDynamic if artifact["model_family"] == "micro" else CompatibleMacroParellelRenormDynamic
    model = model_cls(
        sym_size=int(num_nodes),
        latent_size=int(artifact["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(artifact["hidden_units1"]),
        hidden_units2=int(artifact["hidden_units2"]),
        normalized_state=False,
        device=device,
        is_random=False,
        flow_num_layers=int(artifact["flow_num_layers"]),
        dynamics_num_layers=int(artifact["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=artifact["reduce_dims"],
    ).to(device)
    state_dict = torch.load(artifact["model_path"], map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def prepare_rollout_target(origin_data: np.ndarray, series_index: int = 0, start_step: int = 0, horizon: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    series_data = np.asarray(origin_data, dtype=np.float32)
    if series_data.ndim == 3:
        if not 0 <= int(series_index) < series_data.shape[0]:
            raise IndexError(f"series_index must be in [0, {series_data.shape[0] - 1}]")
        series_data = series_data[int(series_index)]
    if series_data.ndim != 2:
        raise ValueError(f"origin_data must have shape [T, N] or [M, T, N], but got {origin_data.shape}")

    start_step = int(start_step)
    horizon = int(horizon)
    if start_step < 0:
        raise ValueError("start_step must be >= 0")
    if horizon < 1:
        raise ValueError("horizon must be >= 1")
    if start_step + horizon >= len(series_data):
        raise ValueError(f"start_step + horizon must be < {len(series_data)} for the selected series")

    initial_micro_state = series_data[start_step]
    ground_truth = series_data[start_step + 1 : start_step + horizon + 1]
    return initial_micro_state.astype(np.float32), ground_truth.astype(np.float32)


def rollout_scale_prediction(model, initial_micro_state: np.ndarray, scale_id: int, horizon: int = 10) -> Dict:
    initial_tensor = torch.tensor(initial_micro_state, dtype=torch.float32, device=next(model.parameters()).device).unsqueeze(0)
    scale_id = int(scale_id)
    horizon = int(horizon)

    with torch.no_grad():
        macro_state = model.encoding1(initial_tensor, scale_id)[scale_id]
        initial_macro_state = macro_state.detach().cpu().numpy()[0]
        macro_rollout = []
        micro_predictions = []
        for _ in range(horizon):
            macro_state = model._apply_dynamics(macro_state, scale_id, inverse=False)
            macro_rollout.append(macro_state.detach().cpu().numpy()[0])
            micro_predictions.append(model.decoding(macro_state, scale_id).detach().cpu().numpy()[0])

    return {
        "initial_macro_state": np.asarray(initial_macro_state, dtype=np.float32),
        "macro_rollout": np.asarray(macro_rollout, dtype=np.float32),
        "micro_prediction": np.asarray(micro_predictions, dtype=np.float32),
    }


def run_multi_step_prediction(
    project_root: Optional[Union[str, Path]] = None,
    run_name: str = DEFAULT_RUN_NAME,
    data_path: Optional[Union[str, Path]] = None,
    series_index: int = 0,
    start_step: int = 0,
    horizon: int = 10,
    device: Union[str, torch.device] = "cpu",
) -> List[Dict]:
    project_root = find_project_root(Path(project_root) if project_root is not None else None)
    artifacts = list_stage2_scale_artifacts(project_root, run_name=run_name)
    origin_data = load_stage2_origin_data(project_root, Path(data_path) if data_path is not None else None)
    initial_micro_state, ground_truth = prepare_rollout_target(origin_data, series_index=series_index, start_step=start_step, horizon=horizon)

    results = []
    for artifact in artifacts:
        model = build_prediction_model(num_nodes=ground_truth.shape[1], artifact=artifact, device=device)
        rollout = rollout_scale_prediction(model, initial_micro_state, artifact["logical_scale_id"], horizon=horizon)
        prediction = rollout["micro_prediction"]
        results.append(
            {
                **artifact,
                "series_index": int(series_index),
                "start_step": int(start_step),
                "horizon": int(horizon),
                "initial_micro_state": initial_micro_state.copy(),
                "ground_truth": ground_truth.copy(),
                "prediction": prediction,
                "initial_macro_state": rollout["initial_macro_state"],
                "macro_rollout": rollout["macro_rollout"],
                "mae": float(np.mean(np.abs(prediction - ground_truth))),
                "mse": float(np.mean((prediction - ground_truth) ** 2)),
            }
        )
    return results


In [ ]:
def select_variable_indices(num_variables: int, max_variables: int = 4, variable_indices: Optional[Sequence[int]] = None) -> List[int]:
    if variable_indices is not None:
        indices = [int(idx) for idx in variable_indices if 0 <= int(idx) < int(num_variables)]
        return indices[: int(max_variables)]
    return list(range(min(int(num_variables), int(max_variables))))


def plot_scale_prediction(
    result: Dict,
    output_dir: Optional[Union[str, Path]] = None,
    max_variables: int = 4,
    variable_indices: Optional[Sequence[int]] = None,
    dpi: int = 200,
) -> Path:
    ground_truth = np.asarray(result["ground_truth"], dtype=np.float32)
    prediction = np.asarray(result["prediction"], dtype=np.float32)
    indices = select_variable_indices(ground_truth.shape[1], max_variables=max_variables, variable_indices=variable_indices)
    if not indices:
        raise ValueError("No valid variable indices were selected for plotting.")

    if output_dir is None:
        output_dir = Path(result["summary_path"]).resolve().parent / "multi_step_pred"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    figure, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.flatten()
    time_axis = np.arange(1, result["horizon"] + 1)

    for axis, variable_index in zip(axes, indices):
        axis.plot(time_axis, ground_truth[:, variable_index], color="#111827", linewidth=2.2, label="Ground Truth")
        axis.plot(time_axis, prediction[:, variable_index], color="#2563EB", linewidth=2.0, linestyle="--", label="Prediction")
        axis.set_title(f"Variable {variable_index}")
        axis.set_xlabel("Prediction Step")
        axis.set_ylabel("Value")
        axis.grid(alpha=0.25, linestyle="--", linewidth=0.8)

    for axis in axes[len(indices):]:
        axis.axis("off")

    axes[0].legend(loc="best")
    figure.suptitle(
        f"Scale {result['file_scale']} | {result['model_family']} | macro_dim={result['macro_dim']} | 10-step decoded micro prediction",
        fontsize=15,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    saved_path = output_dir / f"scale{result['file_scale']}_multi_step_prediction.png"
    figure.savefig(saved_path, dpi=dpi, bbox_inches="tight")
    if plt.get_backend().lower() != "agg":
        plt.show()
    plt.close(figure)
    return saved_path


def plot_multi_step_predictions(
    results: List[Dict],
    output_dir: Optional[Union[str, Path]] = None,
    max_variables: int = 4,
    variable_indices: Optional[Sequence[int]] = None,
) -> pd.DataFrame:
    summary_rows = []
    for result in results:
        plot_path = plot_scale_prediction(
            result,
            output_dir=output_dir,
            max_variables=max_variables,
            variable_indices=variable_indices,
        )
        summary_rows.append(
            {
                "file_scale": result["file_scale"],
                "model_family": result["model_family"],
                "logical_scale_id": result["logical_scale_id"],
                "macro_dim": result["macro_dim"],
                "series_index": result["series_index"],
                "start_step": result["start_step"],
                "horizon": result["horizon"],
                "mae": result["mae"],
                "mse": result["mse"],
                "plot_path": str(plot_path.resolve()),
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values("file_scale").reset_index(drop=True)
    if output_dir is None and results:
        output_dir = Path(results[0]["summary_path"]).resolve().parent / "multi_step_pred"
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        summary_df.to_csv(output_dir / "multi_step_prediction_summary.csv", index=False)
    return summary_df


In [ ]:
PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "loc_result_stage2" / DEFAULT_RUN_NAME / "multi_step_pred"

MULTI_STEP_RESULTS = run_multi_step_prediction(
    project_root=PROJECT_ROOT,
    run_name=DEFAULT_RUN_NAME,
    data_path=DEFAULT_DATA_PATH,
    series_index=0,
    start_step=0,
    horizon=10,
    device="cuda:3",
)

MULTI_STEP_SUMMARY = plot_multi_step_predictions(
    MULTI_STEP_RESULTS,
    output_dir=OUTPUT_DIR,
    max_variables=4,
)

display(MULTI_STEP_SUMMARY)
MULTI_STEP_SUMMARY
